# Ollama Benchmarks

Run the embedding benchmark first, then reuse one of those cached embedding indexes for the generation benchmark.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))

from eval.embedding_benchmark import EmbeddingBenchmarkConfig, run_embedding_benchmark
from eval.model_benchmark import ModelBenchmarkConfig, run_model_benchmark
from helpers.experiment_models import (
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_GENERATION_MODEL,
    EMBEDDING_MODEL_SWEEP,
    GENERATION_MODEL_SWEEP,
)

os.environ.setdefault("OLLAMA_ENDPOINT", "http://10.0.0.201:8000")
os.environ.setdefault("INPUT_BASE_DIR", str(repo_root / "data" / "evidence" / "mimic_discharge_subset"))

embedding_model_sweep = list(EMBEDDING_MODEL_SWEEP)
generation_model_sweep = list(GENERATION_MODEL_SWEEP)
embedding_generation_model = os.environ.get("BENCHMARK_GENERATION_MODEL", DEFAULT_GENERATION_MODEL)
generation_embedding_model = os.environ.get("BENCHMARK_EMBEDDING_MODEL", DEFAULT_EMBEDDING_MODEL)
sample_size = 30
shared_index_root = repo_root / "output" / "benchmark_indexes"
use_umls = os.environ.get("UMLS_ENABLED", "true").strip().lower() == "true"
schema_guided = os.environ.get("INDEX_SCHEMA_GUIDED", "false").strip().lower() == "true"
mimic_csv = repo_root / "data" / "mimic_iv_note" / "discharge.csv"

print("OLLAMA_ENDPOINT:", os.environ["OLLAMA_ENDPOINT"])
print("embedding_model_sweep:", embedding_model_sweep)
print("generation_model_sweep:", generation_model_sweep)
print("embedding_generation_model:", embedding_generation_model)
print("generation_embedding_model:", generation_embedding_model)
print("shared_index_root:", shared_index_root)
print("schema_guided:", schema_guided)

/Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OLLAMA_ENDPOINT: http://10.0.0.201:8000
embedding_model_sweep: ['qwen3-embedding:0.6b', 'embeddinggemma:latest', 'all-minilm:latest']
generation_model_sweep: ['gemma4:latest', 'qwen3.5:9b', 'medgemma1.5:latest']
embedding_generation_model: gemma4:latest
generation_embedding_model: qwen3-embedding:0.6b
shared_index_root: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/output/benchmark_indexes
schema_guided: False


## Embedding Benchmark

This section builds or reuses one index per embedding model under `output/benchmark_indexes/` while keeping the generation model fixed.

In [2]:
embedding_results = await run_embedding_benchmark(
    EmbeddingBenchmarkConfig(
        input_dir=Path(os.environ["INPUT_BASE_DIR"]),
        output_root=repo_root / "output" / "ollama_embedding_benchmark",
        index_root=shared_index_root,
        generation_model=embedding_generation_model,
        embedding_models=tuple(embedding_model_sweep),
        use_umls=use_umls,
        schema_guided=schema_guided,
        mimic_csv=(mimic_csv if mimic_csv.exists() else None),
        sample_size=sample_size,
    )
)

embedding_results_df = pd.DataFrame(result.__dict__ for result in embedding_results).sort_values(
    ["mean_top_cosine_similarity", "mean_cosine_similarity", "exact_match", "mean_query_seconds"],
    ascending=[False, False, False, True],
)
embedding_results_df

Extracting MIMIC discharge notes from /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/data/mimic_iv_note/discharge.csv into data/evidence/mimic_discharge_subset
Wrote 25 note files to data/evidence/mimic_discharge_subset
Running embedding benchmark for 3 embedding model(s) over 30 question(s)
Building index for embedding model `qwen3-embedding:0.6b`
LlamaIndex index is missing or incompatible:
  - missing /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/output/benchmark_indexes/qwen3-embedding_0_6b/index_manifest.json
Preparing index build for 25 source document(s) into /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/output/benchmark_indexes/qwen3-embedding_0_6b
Index build attempt 1/3
Batch 1/2: inserting 20 document(s)
Indexed batch of 20 document(s)
Batch 2/2: inserting 5 document(s)
Indexed batch of 5 document(s)
Index build completed: /Users/oluwatosinoso/Library/CloudStorage

,embedding_model,index_dir,question_count,build_seconds,total_query_seconds,mean_query_seconds,exact_match,mean_top_retrieval_score,mean_retrieval_score,mean_retrieval_score_margin,mean_top_cosine_similarity,mean_cosine_similarity,mean_cosine_similarity_margin,use_umls,schema_guided,generation_model,provider
0,qwen3-embedding:0.6b,/Users/oluwatosinoso/Library/CloudStorage/OneD...,30,472.68,924.21,30.81,0.0667,0.0,0.0,0.0,0.2372,0.2487,-0.0306,True,False,gemma4:latest,ollama
1,embeddinggemma:latest,/Users/oluwatosinoso/Library/CloudStorage/OneD...,30,459.96,910.36,30.35,0.0333,0.0,0.0,0.0,0.0951,0.0991,-0.0090,True,False,gemma4:latest,ollama
2,all-minilm:latest,/Users/oluwatosinoso/Library/CloudStorage/OneD...,30,468.64,831.02,27.70,0.0333,0.0,0.0,0.0,-0.0095,0.0013,-0.0094,True,False,gemma4:latest,ollama


In [3]:
if not embedding_results_df.empty and "BENCHMARK_EMBEDDING_MODEL" not in os.environ:
    generation_embedding_model = str(embedding_results_df.iloc[0]["embedding_model"])

print("generation_embedding_model:", generation_embedding_model)

generation_embedding_model: qwen3-embedding:0.6b


## Generation Benchmark

This section reuses the selected embedding index from `output/benchmark_indexes/` and compares generation models against the same retrieval layer.

If `BENCHMARK_EMBEDDING_MODEL` is not set, it uses the top row from the embedding benchmark table above.

In [4]:
generation_results = await run_model_benchmark(
    ModelBenchmarkConfig(
        input_dir=Path(os.environ["INPUT_BASE_DIR"]),
        output_root=repo_root / "output" / "ollama_model_benchmark",
        index_root=shared_index_root,
        generation_models=tuple(generation_model_sweep),
        embedding_model=generation_embedding_model,
        use_umls=use_umls,
        schema_guided=schema_guided,
        mimic_csv=(mimic_csv if mimic_csv.exists() else None),
        sample_size=sample_size,
    )
)

generation_results_df = pd.DataFrame(result.__dict__ for result in generation_results).sort_values(
    ["exact_match", "mean_query_seconds"],
    ascending=[False, True],
)
generation_results_df

Extracting MIMIC discharge notes from /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/data/mimic_iv_note/discharge.csv into data/evidence/mimic_discharge_subset
Wrote 25 note files to data/evidence/mimic_discharge_subset
LlamaIndex index is missing or incompatible:
  - source fingerprint changed
Preparing index build for 25 source document(s) into /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/output/benchmark_indexes/qwen3-embedding_0_6b
Index build attempt 1/3
Batch 1/2: inserting 20 document(s)
Indexed batch of 20 document(s)
Batch 2/2: inserting 5 document(s)
Indexed batch of 5 document(s)
Index build completed: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/output/benchmark_indexes/qwen3-embedding_0_6b
Running model benchmark for 3 generation model(s) over 30 question(s)
Querying shared index with generation model `gemma4:latest`
Completed `gemma4:latest`: build=468.17s, m

,generation_model,embedding_model,index_dir,question_count,build_seconds,total_query_seconds,mean_query_seconds,exact_match,use_umls,schema_guided,provider
0,gemma4:latest,qwen3-embedding:0.6b,/Users/oluwatosinoso/Library/CloudStorage/OneD...,30,468.17,933.84,31.13,0.1000,True,False,ollama
1,qwen3.5:9b,qwen3-embedding:0.6b,/Users/oluwatosinoso/Library/CloudStorage/OneD...,30,468.17,901.76,30.06,0.0667,True,False,ollama
2,medgemma1.5:latest,qwen3-embedding:0.6b,/Users/oluwatosinoso/Library/CloudStorage/OneD...,30,468.17,869.92,29.00,0.0333,True,False,ollama
